# RLS Example 2

Access conditioned on record type — the same dimension can mean something different depending on what kind of record is being viewed.

- **Universal roles** (`Country`, `Brand`) — same restriction across all record types, standard roles on their dimension tables.
- **Dataframe prep** — a per-user flag (`Is_Unrestricted_*`), aggregated before any row duplication, plus a row explosion that pairs each conditional dimension with the record type it applies to.
- **Concatenated roles** — `CompanyCode` (Orders only), `IndustryCode` and `CompanySolution` (Opportunities/Pipeline), combining the value with its record type via `concatenated_from` + `is_numeric`.
- **Fallback roles** — `Unrestricted_OppsPipeline`/`Unrestricted_Offers` grant full access on those record types when the precomputed flag says the user has no restriction, even if that same user is restricted on Orders (which has no fallback).

Full write-up: [`docs/CONFIGURATION_REFERENCE.md`](../docs/CONFIGURATION_REFERENCE.md).

### Setup

In [ ]:
workspace = 'DEV_YourWorkspace_Sales'

In [ ]:
dataset = "SALES CLOUD"

In [ ]:
%run Connections

In [ ]:
%run Email notifications

In [ ]:
%run RLS_Management_Functions

In [ ]:
workspace_current = workspace

### Global filters

In [ ]:
global_filters = []  # Example 2 has no Is_Consolidated concept

### Config

### RLS Table

Loads the source table and computes the fallback flags (`Is_Unrestricted_*`).

In [ ]:
source_path = path_silver
source_table = 'rls_sf_sales'

df = spark.read.format("delta").load(f"{source_path}{source_table}")

# ── Fallback: "no restriction in any Opportunity dimension → sees everything" ──
# Mirrors the original dynamic DAX's Case 2/3 logic: if the user has NO value
# in Brand, Country, Industry, or CompanySolution (across ALL their rows), they
# see every Opportunity/Pipeline record. Blanks in this table show up as "" as
# well as NULL, so both need to be covered. For Offers (Case 4), the fallback
# only looks at Brand and Country — Industry and CompanySolution aren't even
# mentioned in that branch of the original DAX.
def _has_value(col_name):
    return (
        F.col(col_name).isNotNull()
        & (F.col(col_name) != "")
        & (F.col(col_name) != "None")
    )

user_flags = (
    df.groupBy("Username")
    .agg(
        F.max(F.when(_has_value("Brand"), 1).otherwise(0)).alias("has_brand"),
        F.max(F.when(_has_value("CountryCode"), 1).otherwise(0)).alias("has_country"),
        F.max(F.when(_has_value("IndustryCode"), 1).otherwise(0)).alias("has_industry"),
        F.max(F.when(_has_value("CompanySolution"), 1).otherwise(0)).alias("has_solution"),
    )
    .withColumn(
        "Is_Unrestricted_478",
        F.when(
            (F.col("has_brand") == 0) & (F.col("has_country") == 0)
            & (F.col("has_industry") == 0) & (F.col("has_solution") == 0),
            F.lit("1")
        ).otherwise(F.lit("0"))
    )
    .withColumn(
        "Is_Unrestricted_6",
        F.when(
            (F.col("has_brand") == 0) & (F.col("has_country") == 0),
            F.lit("1")
        ).otherwise(F.lit("0"))
    )
    .select("Username", "Is_Unrestricted_478", "Is_Unrestricted_6")
)

df = df.join(user_flags, on="Username", how="left")

# CompanyCode only applies to Orders (DocType 7) — just labeling the rows is
# enough, no duplication needed since there's only one possible DocType.
df = df.withColumn(
    "DocType_CompanyCode",
    F.when(F.col("CompanyCode").isNotNull(), F.lit("7"))
)

# IndustryCode applies to Opportunities and Pipeline (DocType 4 and 8) — each
# row with IndustryCode populated DOES need to be duplicated, one copy per DocType.
df_industry_base = df.where(F.col("IndustryCode").isNotNull())
df_industry_4 = df_industry_base.withColumn("DocType_Industry", F.lit("4"))
df_industry_8 = df_industry_base.withColumn("DocType_Industry", F.lit("8"))

# Same pattern for CompanySolution (also DocType 4 and 8)
df_solution_base = df.where(F.col("CompanySolution").isNotNull())
df_solution_4 = df_solution_base.withColumn("DocType_Solution", F.lit("4"))
df_solution_8 = df_solution_base.withColumn("DocType_Solution", F.lit("8"))

# Merge everything into a single dataframe. unionByName with allowMissingColumns
# fills any column that doesn't apply to a given fragment with null.
df = (
    df
    .unionByName(df_industry_4, allowMissingColumns=True)
    .unionByName(df_industry_8, allowMissingColumns=True)
    .unionByName(df_solution_4, allowMissingColumns=True)
    .unionByName(df_solution_8, allowMissingColumns=True)
)

In [ ]:
config = {
    # ── UNIVERSAL (no DocType condition) ────────────────────────────────────
    "Country": {
        "table": "DT_Country", "prefix": "RLS_Country",
        "column": "CountryCode", "source_column": "CountryCode"
    },
    "Brand": {
        "table": "DT_Brand", "prefix": "RLS_Brand",
        "column": "Brand", "source_column": "Brand"
    },

    # ── DOCTYPE-CONDITIONAL, via concatenated roles ─────────────────────────
    "CompanyCode_Orders": {
        "prefix": "RLS_DocType7_CompanyCode",
        "concatenated_from": {
            "CompanyCode":         {"table": "FT_Sales", "column": "CompanyCode"},
            "DocType_CompanyCode": {"table": "FT_Sales", "column": "DocType", "is_numeric": True}
        }
    },
    "IndustryCode_OppsPipeline": {
        "prefix": "RLS_DocType478_IndustryCode",
        "concatenated_from": {
            "IndustryCode":     {"table": "FT_Sales", "column": "IndustryCode"},
            "DocType_Industry": {"table": "FT_Sales", "column": "DocType", "is_numeric": True}
        }
    },
    "CompanySolution_OppsPipeline": {
        "prefix": "RLS_DocType478_CompanySolution",
        "concatenated_from": {
            "CompanySolution":  {"table": "DT_Solution", "column": "Solution"},
            "DocType_Solution": {"table": "FT_Sales",     "column": "DocType", "is_numeric": True}
        }
    },

    # ── FALLBACK: "no restriction on Opportunity/Offers → sees everything" ──
    "Unrestricted_OppsPipeline": {
        "table": "FT_Sales",
        "prefix": "RLS_DocType478_Unrestricted",
        "special": "consolidated",
        "fixed_filter": "[DocType] IN {4, 8}",
        "source_column": "Is_Unrestricted_478",
        "source_value": "1",
        "ignore_cols": [c for c in df.columns if c != "Is_Unrestricted_478"]
    },
    "Unrestricted_Offers": {
        "table": "FT_Sales",
        "prefix": "RLS_DocType6_Unrestricted",
        "special": "consolidated",
        "fixed_filter": "[DocType] = 6",
        "source_column": "Is_Unrestricted_6",
        "source_value": "1",
        "ignore_cols": [c for c in df.columns if c != "Is_Unrestricted_6"]
    },

    # "FullAccess": {
    #     "table": None, "prefix": "RLS_FullAccess", "column": None,
    #     "special": "full_access", "source_column": "FullAccess", "source_value": "ALL"
    # },
}

### AD Users join (UPN validation)

In [ ]:
source_path = path_bronze
source_table = 'ad_userslicences'

df_ad = spark.read.format("delta").load(f"{source_path}{source_table}").select(
    F.col('UserPrincipalName').alias('Username'),
    F.col('UserId').alias('User_ID')
)

#### USER VALIDATION
df = df.join(df_ad, on = 'Username', how='inner')
df = df.where(F.col('User_ID').isNotNull())


## RUN

In [ ]:
df, config = preprocess_concatenated_roles(df, config)

In [ ]:
config["Unrestricted_OppsPipeline"]["ignore_cols"] = list(df.columns)
config["Unrestricted_Offers"]["ignore_cols"] = list(df.columns)

#### 1. Create/update roles

In [ ]:
create_or_replace_roles(config, dataset, workspace, global_filters, rls=df)

#### 2. Clean up unused roles

In [ ]:
#drop_unused_roles(config, dataset, workspace, rls=df)

#### 3. Sync membership (delta, with audit)

In [ ]:
result, summary = run_with_audit(
    update_members_delta,
    dataset=dataset,
    config=config,      
    workspace=workspace,
    env=env,
    rls=df,
    chunk_size=1000,             
    alert_threshold=0.05,
    critical_drop_threshold=0.5,
    audit_path=path_control,     
    alert_to="your-team@yourcompany.com",
    skip_members=[]
)


In [ ]:
# # One-time rename of columns on the already-existing tables
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN entorno TO environment")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN modelo TO model")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN miembros_antes TO members_before")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN miembros_despues TO members_after")
# # (same for rls_change_log: entorno → environment, modelo → model)